### Helper.py

In [1]:
import re
import yaml
import json
import os

class LiteralString(str):
    pass
def literal_representer(dumper,value):
    return dumper.represent_scalar('tag:yaml.org,2002:str', value, style='|')

# Register custom representer
yaml.add_representer(LiteralString, literal_representer)

class Helper:
    @staticmethod
    def load_file(filepath: str) -> str:
        with open(filepath, "r", encoding="utf-8") as file:
            return file.read()
    @staticmethod
    def load_yaml(filepath: str) -> dict:
        with open(filepath, "r", encoding="utf-8") as f:
            return yaml.safe_load(f)
    @staticmethod
    def load_json(filepath: str) -> dict:
        with open(filepath, "r", encoding="utf-8") as f:
            return json.load(f)
    @staticmethod
    def save_yaml(newconfig,filepath: str):
        with open(filepath,"w",encoding="utf-8") as file:
            return yaml.dump(newconfig,file,sort_keys=False)
    @staticmethod
    def fop(num: float) -> float:
        return float(f"{num:.1f}")
    @staticmethod
    def prettyjson(txt:str) -> str:
        return str(json.dumps(txt,indent=4, ensure_ascii=False))
    @staticmethod
    def to_literal(value):
        if isinstance(value,str) and "\n" in value:
            return LiteralString(value)
        return value
    @staticmethod
    def deep_literal_transform(data):
        if isinstance(data, dict):
            return {k: Helper.deep_literal_transform(v) for k,v in data.items()}
        if isinstance(data, list):
            return [Helper.deep_literal_transform(i) for i in data]
        return Helper.to_literal(data)

### Llmcaller.py

In [ ]:
import asyncio
from pydantic import BaseModel, Field, ValidationError
from typing import Dict

class CriterionScore(BaseModel):
    score: int = Field(ge=0, le=5)
    feedback: str

class SectionEvaluation(BaseModel):
    section: str
    scores: Dict[str, CriterionScore]
    session_feedback: str

class LlmCaller(Helper):
    def __init__(self):
        self.client    = genai.Client(api_key="AIzaSyAR...3dHAWb5urRk")
        self.model_cfg = self.load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\model.yaml")
        self.model     = self.model_cfg["model"]["generation_model"]
        self.usd2bath  = Helper.load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\global.yaml")['currency']['USD_to_THB']
        self.log_digit = Helper.load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\global.yaml")['logging']['logging_round_digit']
    def extract_json(text: str) -> dict:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise ValueError("NO_JSON_OBJECT_FOUND")
        return json.loads(match.group())
    def _parse(self, resp):
        text = resp.text.strip()
        text = re.sub(r"^```json|```$", "", text).strip()
        try:
            return json.loads(text)
        except json.JSONDecodeError as e:
            pass        
        try:
            return self.extract_json(text)
        except Exception as e:
            raise ValueError(f"INVALID_JSON::{text}") from e
    def _call_raw(self, prompt: str):
        resp = self.client.models.generate_content(
            model=self.model,
            contents=prompt
        )
        parsed = self._parse(resp)
        return parsed, resp
    def _validate(self, raw_output:dict)->SectionEvaluation:
        return SectionEvaluation.model_validate(raw_output)
    def _repair_prompt(self, error_msg: str) -> str:
        return f"""
                Your previous response was INVALID.
                Validation error:
                {error_msg}
                STRICT RULES:
                - Return JSON only
                - No markdown
                - No explanation
                - Follow schema exactly
                - Section name must start with a capital letters (e.g. "Education")
                Expected format:
                {{
                    "section": "<Section_name>",
                    "scores": {{
                        "<criterion>": {{ 
                            "score": 0-5, 
                            "feedback": "string" 
                        }}
                    }},
                    "session_feedback":"string"
                }}
        """
    def call(self,prompt:str, max_retry:int = 3):
        last_error = None
        repair_prompt = "\n"
        for attemp in range(max_retry):
            final_prompt = repair_prompt + prompt
            # print(f"final_prompt attemp : {attemp} -> \n {final_prompt}")
            # print(f"Output -> \n{output}")
            try:
                output, raw = self._call_raw(final_prompt)
                validated   = self._validate(output)
                # print('Status : 1')
                return validated.model_dump(),raw
            except (ValidationError, ValueError) as e:
                last_error = str(e)
                repair_prompt = self._repair_prompt(last_error)
                # print('Status : 0')
                print("Output error recall again ...")
            finally:
                print('='*100)
        return {
            "section":"UNKNOW",
            "scores":{}
        },raw
    async def call_async(self, prompt: str):
        return await asyncio.to_thread(self.call, prompt)

### PromptBuilder.py

In [3]:
from google import genai
import json
import os
import yaml


class BasePromptBuilder(Helper):
    '''
    PromptBuilder v3 : PromptBuilder + Session,Global feedback + PromptSplit
    '''
    base_dir = r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\prompts"         # Prompts .yaml config file folder path
    def __init__(self, section, criteria, targetrole, cvresume, include_fewshot: bool = True, output_lang = "en"):
        self.section         = section
        self.criteria        = criteria[::-1]
        self.targetrole      = targetrole
        self.cvresume        = cvresume
        self.include_fewshot = include_fewshot
        self.output_lang     = output_lang

        self.global_config   = self.load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\prompts\base.yaml")
        self.section_config  = self.load_yaml(f"{self.base_dir}/{self.section.lower()}.yaml")
        self.number_of_words = self.load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\global.yaml")["output"]["number_of_words"]
        self.criteria_cfg    = self.section_config['criteria']
    def _build_response_template(self):
        return {
            "section": self.section,
            "scores": {
                c: {"score": 0, "feedback": ""} for c in self.criteria
            },
            "session_feedback":""
        }
    
    def _build_criteria_block(self) -> str:
        blocks = []
        for crit in self.criteria:
            block = f"- {crit}\n"
            for i in [5,3,1]:
                block += f"  score {i} :\n"
                x = f"score{i}"
                block += "\n".join(
                    "    " + line
                    for line in self.criteria_cfg[crit][x].splitlines()
                ) + "\n"
            blocks.append(block)
        return "".join(blocks)

    def build(self):
        config_role       = self.global_config['role']['role1']
        config_task       = self.section_config['task']['task1']
        config_lang       = self.global_config['Language_output_style'][self.output_lang]
        config_expected   = self.section_config['expected_content'][self.section]
        criteria_block    = self._build_criteria_block()
        config_example    = self.section_config['output_guidelines'][self.section]
        config_scale      = self.global_config['scale']['score1']
        config_feedback   = self.global_config['feedback']['globalfeedback']

        prompt_role       = f"Role :\n{config_role}\n\n"
        prompt_task       = f"Task :\n{config_task}\n"
        prompt_lang       = f"Output language instruction :\n{config_lang}\n"
        prompt_expected   = f"Expected :\n{config_expected}\n"
        prompt_criteria   = f"Criteria :\n{criteria_block}\n"
        prompt_scale      = f"Scale :\n{config_scale}\n"
        prompt_feedback   = f"Session feedback :\n{config_feedback}\n\n"
        prompt_Op_example = f"Output guideline :\n{config_example}\n\n"
        prompt_Op_format  = f"Output format :\n{json.dumps(self._build_response_template(), indent=2)}\n\n"
        prompt_cvresume   = f"CV/Resume :\n{self.cvresume}\n"

        prompt = (
            prompt_role + prompt_task + prompt_lang
            + prompt_expected + prompt_criteria + prompt_scale + prompt_feedback 
            + prompt_Op_example + prompt_Op_format + prompt_cvresume 
        )

        prompt = prompt.replace("<section_name>", self.section)
        prompt = prompt.replace("<targetrole>", self.targetrole)
        prompt = prompt.replace("<number_of_words>", str(self.number_of_words))

        return prompt

### aggregator.py

In [ ]:
from datetime import datetime,timezone,timedelta
import copy
import json

class SectionScoreAggregator(Helper):
    def __init__(self):
        self.config          = self.load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\weight.yaml")         # config
    def aggregate(self,llm_output:dict):
        # print(f"llm_output ->\n{llm_output}")
        self.llm_output      = llm_output             # op
        self.section         = llm_output["section"]  # Get section
        self.section_weights = self.config["weights"][self.section]  # config["weights"][section_key][criteria]
        ddict = {}
        total = 0.0
        # Protect multiple mutation when we run more than one time
        scores_copy = copy.deepcopy(self.llm_output["scores"])
        for criteria, body in scores_copy.items():
            raw = body["score"]
            w   = self.section_weights[criteria]
            weighted = raw / 5 * w
            body["score"] = weighted
            ddict[criteria] = body
            total = total + weighted
        return {
            "section": self.section,
            "total_score":total,
            "scores":ddict,
            "session_feedback":self.llm_output['session_feedback']
        }
    
class GlobalAggregator(LlmCaller,Helper):
    def __init__(self,SectionScoreAggregator_output:list,output_lang):
        super().__init__()    # Run Llmcaller class 
        self.section_outputs = SectionScoreAggregator_output
        self.timestamp       = str(datetime.now(tz=(timezone(timedelta(hours=7)))))
        self.model_config    = Helper.load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\model.yaml")     # should include model name
        self.weight_config   = Helper.load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\weight.yaml")    # includes weights + version
        self.prompt_config   = Helper.load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\prompt.yaml")    # includes prompt version
        self.config_lang     = self.prompt_config['Language_output_style'][output_lang]
        
    def fn1(self):
        weights = self.weight_config["weights"]
        contribution = {}
        total = 0.0
        for section_data in self.section_outputs:
            section_name    = section_data["section"]
            total_score     = section_data["total_score"]
            section_weight  = weights[section_name]["section_weight"]
            section_contrib = total_score * section_weight
            contribution[section_name] = {
                "section_total": total_score,
                "section_weight": section_weight,
                "contribution": Helper.fop(section_contrib)
            }
            total = total + section_contrib
        return {
            "final_resume_score":Helper.fop(total),
            "section_contribution":contribution,
            "globalfeedback":self.parse
        }
    
    def fn2(self):
        details = {}
        for section_data in self.section_outputs:
            # print(f"section_data->\n{section_data}")
            details[section_data["section"]] = {
                'total_score':section_data['total_score'],
                'scores':section_data['scores'],
                'session_feedback':section_data['session_feedback']
            }
        prompt = self.prompt_config['feedback']['globalfeedback']
        # print(f"prompt->\n{prompt}")
        self.parse,_ = self._call_raw(prompt)
        # print(self.parse)
        return details
    
    def fn3(self):
        return {
            "model_name": self.model_config['model']['generation_model'],
            "timestamp": self.timestamp,
            "weights_version": self.weight_config.get("version", "unknown"),
            "prompt_version": self.prompt_config.get("version", "unknown")
        }
    
    def fn0(self):
        detail_part     = self.fn2()
        conclution_part = self.fn1()
        metadata_part   = self.fn3()
        return {
            "conclution":conclution_part,
            "section_detail":detail_part,
            "metadata":metadata_part
        }
    



<hr>

In [5]:
caller    = LlmCaller()
agg       = SectionScoreAggregator()

In [6]:
mock_data = Helper.load_json(r"C:\Users\TunKedsaro\Desktop\CVResume\src\mock\resume4.json")

In [126]:
p1 = BasePromptBuilder(
    section     = "Profile",
    criteria    = ["Completeness", "ContentQuality"],
    targetrole  = "data scientist",
    cvresume    = mock_data,
    output_lang = "en"
)
prompt1 = p1.build()

p2 = BasePromptBuilder( 
    section     = "Summary", 
    criteria    = ["Completeness", "ContentQuality","Grammar","Length","RoleRelevance"],
    targetrole  = "data scientist",
    cvresume    = mock_data,
    output_lang = "en"
)
prompt2 = p2.build()

p3 = BasePromptBuilder( 
    section     = "Education", 
    criteria    = ["Completeness","RoleRelevance"],
    targetrole  = "data scientist",
    cvresume    = mock_data,
    output_lang = "en"
)
prompt3 = p3.build()

p4 = BasePromptBuilder( 
    section     = "Experience", 
    criteria    = ["Completeness", "ContentQuality","Grammar","Length","RoleRelevance"],
    targetrole  = "data scientist",
    cvresume    = mock_data,
    output_lang = "en"
)
prompt4 = p4.build()

p5 = BasePromptBuilder( 
    section     = "Activities", 
    criteria    = ["Completeness", "ContentQuality","Grammar","Length"],
    targetrole  = "data scientist",
    cvresume    = mock_data,
    output_lang = "en"
)
prompt5 = p5.build()

p6 = BasePromptBuilder( 
    section     = "Skills", 
    criteria    = ["Completeness","Length","RoleRelevance"],
    targetrole  = "data scientist",
    cvresume    = mock_data,
    output_lang = "en"
)
prompt6 = p6.build()

op1,raw1 = caller.call(prompt1)
op2,raw2 = caller.call(prompt2)
op3,raw3 = caller.call(prompt3)
op4,raw4 = caller.call(prompt4)
op5,raw5 = caller.call(prompt5)
op6,raw6 = caller.call(prompt6)

<hr>

In [127]:
class SectionPresenceDetector:
    def __init__(self,cv_json:dict):
        self.cv = cv_json
    def is_provided(self,value)->bool:
        '''
        Detect items. Is any data is exist?
        '''
        if value is None:
            print("Op1")
            return False
        if isinstance(value,str):
            print("Op2")
            return value.strip() != ""
        if isinstance(value,list):
            print("Op3")
            return len(value) > 0
        if isinstance(value,dict):
            print("Op4")
            found = False
            for v in value.values():
                # print(v)
                if v not in ("",None,[],{}):
                    found = True
                    break
            return found
    # def detect(self):
    #     x1 = self.is_provided(self.cv['Profile'])
    #     x2 = self.is_provided(self.cv['Summary'])
    #     x3 = self.is_provided(self.cv['Education'])
    #     x4 = self.is_provided(self.cv['Experience'])
    #     x3 = self.is_provided(self.cv['Activities'])
    #     x6 = self.is_provided(self.cv['Skills'])
    #     print(x4)
    def detect(self):
        return {
            "Profile":self.is_provided(self.cv.get('Profile')),
            "Summary":self.is_provided(self.cv.get('Summary')),
            "Education":self.is_provided(self.cv.get('Education')),
            "Experience":self.is_provided(self.cv.get('Experience')),
            "Activities":self.is_provided(self.cv.get('Activities')),
            "Skills":self.is_provided(self.cv.get('Skills')),
        }

In [128]:
mock_data = Helper.load_json(r"C:\Users\TunKedsaro\Desktop\CVResume\src\mock\resume4.json")
spd1 = SectionPresenceDetector(mock_data)
provide_detect = spd1.detect()

Op4
Op4
Op3
Op3
Op3
Op4


In [129]:
provide_detect

{'Profile': True,
 'Summary': True,
 'Education': True,
 'Experience': True,
 'Activities': False,
 'Skills': True}

In [130]:
op1

{'section': 'Profile',
 'scores': {'ContentQuality': {'score': 1,
   'feedback': "The profile section lacks a clear professional title to define the candidate's identity."},
  'Completeness': {'score': 3,
   'feedback': "The section provides contact information and links, but it omits the candidate's professional title or status."}},
 'session_feedback': 'The profile provides contact details and links, but lacks a professional title, hindering immediate clarity for the Data Scientist role.'}

<hr>
<hr>

### start here

In [385]:
op1 = {
    'section': 'Profile',
    'scores': {
        'ContentQuality': {
            'score': 2,
            'feedback': "xxx"},
        'Completeness': {
            'score': 3,
            'feedback': "yyy"}
        },
    'session_feedback': 'zzz'
}

In [386]:
op3 = {
    'section': 'Education',
    'scores': {
        'RoleRelevance': {
            'score': 5,
            'feedback': 'abc'},
        'Completeness': {
        'score': 5,
        'feedback': "def"}
        },
    'session_feedback': 'ghi'
}

In [387]:
def load_yaml(filepath: str) -> dict:
    with open(filepath, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

In [388]:
config          = load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\weight.yaml") 
llm_output      = op1
section         = op1['section']
section_weights = config["weights"][section]
llm_output

{'section': 'Profile',
 'scores': {'ContentQuality': {'score': 2, 'feedback': 'xxx'},
  'Completeness': {'score': 3, 'feedback': 'yyy'}},
 'session_feedback': 'zzz'}

In [389]:
section_weights

{'Completeness': 10,
 'ContentQuality': 10,
 'Grammar': 0,
 'Length': 0,
 'RoleRelevance': 0,
 'section_weight': 0.5}

In [390]:
ddict = {}
total = 0.0
scores_copy = copy.deepcopy(llm_output["scores"])
for criteria,body in scores_copy.items():
    print(f"criteria->{criteria}")
    print(f"body->{body}")
    raw = body["score"]             # score ดิบๆ ที่ออกมาจาก LLM
    print(f" - raw -> {raw}") 
    w = section_weights[criteria]   # weight ที่ set ใน weight.yaml เพื่อ scale เต็มๆ
    print(f" - w   -> {w}")
    weighted = raw / 5 * w          # raw / normalize max score * weight scale
    print(f" - weighted -> {weighted}")  
    body["score"] = weighted
    print(f" - body['score'] -> {body['score']}")  
    ddict[criteria] = body
    total = total + weighted
    print(f"total -> {total}")
    print()
ddict

criteria->ContentQuality
body->{'score': 2, 'feedback': 'xxx'}
 - raw -> 2
 - w   -> 10
 - weighted -> 4.0
 - body['score'] -> 4.0
total -> 4.0

criteria->Completeness
body->{'score': 3, 'feedback': 'yyy'}
 - raw -> 3
 - w   -> 10
 - weighted -> 6.0
 - body['score'] -> 6.0
total -> 10.0



{'ContentQuality': {'score': 4.0, 'feedback': 'xxx'},
 'Completeness': {'score': 6.0, 'feedback': 'yyy'}}

In [391]:
{
            "section": section,
            "total_score":total,
            "scores":ddict,
            "session_feedback":llm_output['session_feedback']
        }

{'section': 'Profile',
 'total_score': 10.0,
 'scores': {'ContentQuality': {'score': 4.0, 'feedback': 'xxx'},
  'Completeness': {'score': 6.0, 'feedback': 'yyy'}},
 'session_feedback': 'zzz'}

<hr>

In [392]:
def aggregate(llm_output:dict):
    # print(f"llm_output ->\n{llm_output}")
    llm_output      = llm_output                  # op
    section         = llm_output["section"]       # Get section
    section_weights = config["weights"][section]  # config["weights"][section_key][criteria]
    ddict = {} 
    total = 0.0
    scores_copy = copy.deepcopy(llm_output["scores"])  # Protect multiple mutation when we run more than one time
    for criteria,body in scores_copy.items():
        print(f"criteria->{criteria}")
        print(f"body->{body}")
        raw = body["score"]             # score ดิบๆ ที่ออกมาจาก LLM
        print(f" - raw -> {raw}") 
        w = section_weights[criteria]   # weight ที่ set ใน weight.yaml เพื่อ scale เต็มๆ
        print(f" - w   -> {w}")
        weighted = raw / 5 * w          # raw / normalize max score * weight scale
        print(f" - weighted -> {weighted}")  
        body["score"] = weighted
        print(f" - body['score'] -> {body['score']}")  
        ddict[criteria] = body
        total = total + weighted
        print(f"total -> {total}")
        print()
    return {
            "section": section,
            "total_score":total,
            "scores":ddict,
            "session_feedback":llm_output['session_feedback']
        }

In [393]:
s1 = aggregate(op1)
s2 = aggregate(op3)

criteria->ContentQuality
body->{'score': 2, 'feedback': 'xxx'}
 - raw -> 2
 - w   -> 10
 - weighted -> 4.0
 - body['score'] -> 4.0
total -> 4.0

criteria->Completeness
body->{'score': 3, 'feedback': 'yyy'}
 - raw -> 3
 - w   -> 10
 - weighted -> 6.0
 - body['score'] -> 6.0
total -> 10.0

criteria->RoleRelevance
body->{'score': 5, 'feedback': 'abc'}
 - raw -> 5
 - w   -> 10
 - weighted -> 10.0
 - body['score'] -> 10.0
total -> 10.0

criteria->Completeness
body->{'score': 5, 'feedback': 'def'}
 - raw -> 5
 - w   -> 10
 - weighted -> 10.0
 - body['score'] -> 10.0
total -> 20.0



In [394]:
s1

{'section': 'Profile',
 'total_score': 10.0,
 'scores': {'ContentQuality': {'score': 4.0, 'feedback': 'xxx'},
  'Completeness': {'score': 6.0, 'feedback': 'yyy'}},
 'session_feedback': 'zzz'}

In [395]:
s2

{'section': 'Education',
 'total_score': 20.0,
 'scores': {'RoleRelevance': {'score': 10.0, 'feedback': 'abc'},
  'Completeness': {'score': 10.0, 'feedback': 'def'}},
 'session_feedback': 'ghi'}

<hr>

In [396]:
section_outputs = [s1,s2]
timestamp       = str(datetime.now(tz=(timezone(timedelta(hours=7)))))
model_config    = load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\model.yaml")     # should include model name
weight_config   = load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\weight.yaml")    # includes weights + version
prompt_config   = load_yaml(r"C:\Users\TunKedsaro\Desktop\CVResume\src\config\prompt.yaml") 
config_lang     = prompt_config['Language_output_style']["en"]

In [397]:
section_outputs

[{'section': 'Profile',
  'total_score': 10.0,
  'scores': {'ContentQuality': {'score': 4.0, 'feedback': 'xxx'},
   'Completeness': {'score': 6.0, 'feedback': 'yyy'}},
  'session_feedback': 'zzz'},
 {'section': 'Education',
  'total_score': 20.0,
  'scores': {'RoleRelevance': {'score': 10.0, 'feedback': 'abc'},
   'Completeness': {'score': 10.0, 'feedback': 'def'}},
  'session_feedback': 'ghi'}]

In [398]:
weights = weight_config["weights"]
contribution = {}
total = 0.0

In [399]:
section_status = provide_detect
section_status

{'Profile': True,
 'Summary': True,
 'Education': True,
 'Experience': True,
 'Activities': False,
 'Skills': True}

In [400]:
total = 0
for section_data in section_outputs:
    section_name = section_data["section"]
    total_score  = section_data["total_score"]
    total = total + total_score
print(total)

30.0


<hr>
<hr>

In [383]:
# def fn1
contribution = {}
total = 0.0
for section_data in section_outputs:
    print(section_data)
    section_name = section_data["section"]
    print(f" - section_name->{section_name}")
    total_score  = section_data["total_score"]
    print(f" - total_score->{total_score}")
    section_weight = weights[section_name]["section_weight"]
    print(f" - section_weight->{section_weight}")
    section_contrib = total_score * section_weight
    print(f" - total_score ({total_score}) x section_weight ({section_weight})= section_contrib->{section_contrib}")
    contribution[section_name] = {
                "section_total": total_score,
                "section_weight": section_weight,
                "contribution": Helper.fop(section_contrib)
            }
    print(contribution)
    total = total + section_contrib
    print(f" - total->{total}")

    print()


{'section': 'Profile', 'total_score': 20.0, 'scores': {'ContentQuality': {'score': 10.0, 'feedback': 'xxx'}, 'Completeness': {'score': 10.0, 'feedback': 'yyy'}}, 'session_feedback': 'zzz'}
 - section_name->Profile
 - total_score->20.0
 - section_weight->0.1
 - total_score (20.0) x section_weight (0.1)= section_contrib->2.0
{'Profile': {'section_total': 20.0, 'section_weight': 0.1, 'contribution': 2.0}}
 - total->2.0

{'section': 'Education', 'total_score': 20.0, 'scores': {'RoleRelevance': {'score': 10.0, 'feedback': 'abc'}, 'Completeness': {'score': 10.0, 'feedback': 'def'}}, 'session_feedback': 'ghi'}
 - section_name->Education
 - total_score->20.0
 - section_weight->0.2
 - total_score (20.0) x section_weight (0.2)= section_contrib->4.0
{'Profile': {'section_total': 20.0, 'section_weight': 0.1, 'contribution': 2.0}, 'Education': {'section_total': 20.0, 'section_weight': 0.2, 'contribution': 4.0}}
 - total->6.0



In [382]:
# def fn1-2
contribution = {}
weights = weight_config["weights"]
total = 0.0
effective_weight_sum = 0.0
for section_data in section_outputs:
    print(f"section_data -> {section_data}")
    section_name = section_data["section"]
    print(f" - section_name->{section_name}")
    section_score  = section_data["total_score"]      # section_score คือ max_score
    print(f" - section_score->{section_score}")
    section_weight = weights[section_name]["section_weight"] # ตัวถ่วงนน ของ แต่ละ session
    print(f" - section_weights->{section_weight}")

    status = section_status.get(section_name,False)    # Status of presence
    print(f" - section->{status}")                 #
    if not status: # If not(Fasle) -> True skip it
        continue #
    effective_weight_sum = effective_weight_sum + section_score 
    print(f" - effective_weight_sum -> {effective_weight_sum}")
    contribution[section_name] = {
        "section_total":section_score,
        "section_weight":section_weight,
        "contribution":Helper.fop(section_score*section_weight) # contribution = section_score x weight of session
    }
    total = total + (section_score * section_weight)
    print(f" - total->{total}")
final = total / effective_weight_sum if effective_weight_sum > 0 else 0
print(f"Final -> {final}")
contribution

section_data -> {'section': 'Profile', 'total_score': 20.0, 'scores': {'ContentQuality': {'score': 10.0, 'feedback': 'xxx'}, 'Completeness': {'score': 10.0, 'feedback': 'yyy'}}, 'session_feedback': 'zzz'}
 - section_name->Profile
 - section_score->20.0
 - section_weights->0.1
 - section->True
 - effective_weight_sum -> 20.0
 - total->2.0
section_data -> {'section': 'Education', 'total_score': 20.0, 'scores': {'RoleRelevance': {'score': 10.0, 'feedback': 'abc'}, 'Completeness': {'score': 10.0, 'feedback': 'def'}}, 'session_feedback': 'ghi'}
 - section_name->Education
 - section_score->20.0
 - section_weights->0.2
 - section->True
 - effective_weight_sum -> 40.0
 - total->6.0
Final -> 0.15


{'Profile': {'section_total': 20.0,
  'section_weight': 0.1,
  'contribution': 2.0},
 'Education': {'section_total': 20.0,
  'section_weight': 0.2,
  'contribution': 4.0}}